# Overview

Modelers at the CDC's FluSight Forecast Hub are tasked with producing unconditional probabilistic forecasts for weekly influenza hospitalizations across the United States. This is a critical public health task, as these forecasts help inform resource allocation and policy decisions during flu season. The challenge is to create forecasts that characterize uncertainty across all reasonable future scenarios, not just a limited set of conditions.

This task aims to develop a superhuman forecasting model for this problem. The input data is a time series of historical flu hospitalizations and related signals. The output must be a set of quantile forecasts for all 50 states, Washington DC, and Puerto Rico, across multiple time horizons. The goal is to develop a model that is more accurate and better calibrated than those produced by leading human experts.


## **Problem Statement & Your Deliverable**

Your primary task is to create a forecasting model that predicts **probabilistic forecasts** of **Total Influenza Hospital Admissions** for every US state and jurisdiction. The goal is to create a model that achieves the lowest possible **Weighted Interval Score (WIS)** over a rolling-window evaluation.

Your deliverable is a single Python function, `fit_and_predict_fn`, that takes in training and test data and returns a pandas DataFrame containing the required quantile predictions.

---

### **Implementation Details**

**1. Function Signature & Output Requirements**

Your forecasting model **must** be encapsulated within a function named `fit_and_predict_fn` that adheres to the following signature.

*   **Function Signature:**
    ```python
    def fit_and_predict_fn(
        train_x: pd.DataFrame,
        train_y: pd.Series,
        test_x: pd.DataFrame,
    ) -> pd.DataFrame:
        # Your code here to train your model and generate quantile predictions
        return test_y_hat_quantiles
    ```

*   **Output Format:** Your function **must return a pandas DataFrame** with the following properties:
    *   The **index must match** the index of the input `test_x` DataFrame.
    *   The **columns must be named** according to the required quantiles (e.g., `quantile_0.01`, `quantile_0.5`, `quantile_0.975`).
    *   **Crucial Constraint:** The predicted quantiles for any given row must be **monotonically increasing**.

**2. Dataset Description**

The following data objects are available for you to use:

*   **Primary Training Data:**
    *   `train_x`: A DataFrame containing historical feature data.
    *   `train_y`: A Series containing the historical target values (`Total Influenza Admissions`).
*   **Historical Augmentation Data:**
    *   `ilinet_hhs`, `ilinet`, `ilinet_state`: DataFrames containing ~20 years of historical Influenza-Like Illness (ILI) data, available only for dates before `2022-10-15`.
*   **Reference & Example Data:**
    *   `locations`: A DataFrame with geographic and population data.
    *   `sample_submission_df`: A DataFrame showing the correct final output format.
    *   `example_train_x`, `example_train_y`, `example_test_x`: Small example DataFrames demonstrating the structure of the data passed into your function.
    *   `example_reference_date`: A sample forecast date (`2025-01-18`) for context.

**Feature and Column Definitions:**
*   `target_end_date`: The Saturday of the epiweek for which data is reported.
*   `location_name`: The full name of the US state or territory.
*   `location`: The numeric FIPS code for the location.
*   `population`: The total population of the location.
*   `Total Influenza Admissions`: **The target variable.** This data is only available from late 2020 onwards.

**3. Augmenting Training Data with Historical ILINet Data**
The core challenge is the limited history of the target variable. To overcome this, you are provided with ~20 seasons of historical ILINet data. While not the same target, it is highly correlated and captures the essential seasonal dynamics of influenza. The key is to find a way to make this historical data useful for predicting the modern target.
You may explore one of the following strategies, a combination of the two, or come up with a new strategy to incorporate the historical data.

**Strategy 1: Standardize and Combine**
1.  Apply a standardization method to both datasets independently to make the "shape" of the seasons comparable.
2.  Treat the standardized historical ILINet data as additional, independent flu seasons and append them to your training data.
3.  Train your model on this combined "library" of seasons.

**Strategy 2: Learn a Transformation**
1.  Identify the date range where both the target and the historical ILINet data overlap.
2.  Use this period to learn a statistical transformation that maps the ILINet data onto the same scale as the `Total Influenza Admissions` data.
3.  Apply this transformation to the entire 20-year history of ILINet data to create a "synthetic" history for your target variable.
4.  Train your model on this new, augmented training set.

**4. Key Considerations**

*   **Time Series Awareness:** Use your expertise in handling seasonality, trends, and lags.
*   **Calibration:** Ensure your predicted quantiles are well-calibrated.
*   **Logging:** Configure any model you train to be as quiet as possible (e.g., set `verbose=0`). Do not suppress critical Python warnings or error tracebacks.

**5. Detailed Instructions:**
*   An expert has instructed you to implement the below method for this forecasting task. You may make minor improvements to this method, but the original core principles of the method **MUST** be maintained.

An SIR model with unknown case ascertainment, basic reproduction number, population immunity and a splined effective reproduction number is used to model seasonal influenza dynamics in a given season. Across-season trends ('hyperparameters') in the SIR model's parameters are derived by wrapping it in an across-season Bayesian hierarchical model. Hyperparameters are used as priors when forecasting the current season. Disease model integrated in C++ and bound to Python with pybind11, Bayesian hierarchical posterior probability coded in raw Python and sampled using the ensemble sampler of Goodman and Weare available in `emcee` (motivation: computationally inefficient but amazingly robust).

**Before you write your code,** add a comment block at the top of the cell and explicitly list the 3-4 core principles of the method described. Then, write your implementation, ensuring it strictly adheres to these principles.\n

In [ ]:
!pip install -q emcee pybind11 scipy

In [ ]:
import warnings
import numpy as np
import pandas as pd

TARGET_STR = 'Total Influenza Admissions'

# --- Configuration Constants ---

QUANTILES = [
    0.01,
    0.025,
    0.05,
    0.1,
    0.15,
    0.2,
    0.25,
    0.3,
    0.35,
    0.4,
    0.45,
    0.5,
    0.55,
    0.6,
    0.65,
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.95,
    0.975,
    0.99,
]
horizons = [0, 1, 2, 3]


def _create_fold_scaffold(
    reference_date: pd.Timestamp,
    horizons: list,
    location_codes: np.ndarray,
    locations_df: pd.DataFrame,
) -> pd.DataFrame:
  """Creates the feature scaffold for a single forecast date."""
  all_combinations = pd.MultiIndex.from_product(
      [[reference_date], horizons, location_codes],
      names=['reference_date', 'horizon', 'location'],
  )
  scaffold = pd.DataFrame(index=all_combinations).reset_index()
  scaffold['target_end_date'] = scaffold.apply(
      lambda row: row['reference_date'] + pd.Timedelta(weeks=row['horizon']),
      axis=1,
  )
  return scaffold.merge(locations_df, on='location', how='left')


np.seterr(over='raise')

In [ ]:
example_reference_date = "2021-01-09"
example_train_end_date = "2021-01-02"

In [ ]:
# [exclude_from_prompt]

INPUT_DIR = "./datasets/cdc-flu-forecasting"
WORKING_DIR = "./"

dataset = pd.read_csv(f"{INPUT_DIR}/dataset.csv")
flusight_2023_2024 = pd.read_csv(f"{INPUT_DIR}/top_flusight_2023_2024.csv")
flusight_2024_2025 = pd.read_csv(f"{INPUT_DIR}/top_flusight_2024_2025.csv")
ilinet_hhs = pd.read_csv(f"{INPUT_DIR}/ilinet_hhs_before_20221015.csv")
ilinet = pd.read_csv(f"{INPUT_DIR}/ilinet_before_20221015.csv")
ilinet_state = pd.read_csv(f"{INPUT_DIR}/ilinet_state_before_20221015.csv")
locations = pd.read_csv(f"{INPUT_DIR}/locations.csv")
sample_submission_df = pd.read_csv(f"{INPUT_DIR}/sample_submission.csv")

REQUIRED_CDC_LOCATIONS = ['01', '02', '04', '05', '06', '08', '09', '10', '11', '12', '13',
       '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
       '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36',
       '37', '38', '39', '40', '41', '42', '44', '45', '46', '47', '48',
       '49', '50', '51', '53', '54', '55', '56', '72']

locations = locations[locations['location'].isin(REQUIRED_CDC_LOCATIONS)]
locations['location'] = locations['location'].astype(int)
location_codes = locations["location"].unique()


train = dataset[dataset["target_end_date"] <= example_train_end_date]
example_train_x = train.drop(columns=[TARGET_STR])
example_train_y = train[TARGET_STR]
example_test_x = _create_fold_scaffold(
    pd.Timestamp(example_reference_date),
    horizons,
    location_codes,
    locations_df=locations,
)
dataset["target_end_date"] = pd.to_datetime(dataset["target_end_date"]).dt.date

In [ ]:
print(sample_submission_df.info())
print(sample_submission_df.head().to_markdown())

In [ ]:
print(locations.head().to_markdown())

In [ ]:
print(example_train_x.info())
print(example_train_x.head().to_markdown())

In [ ]:
print(example_train_y.info())
print(example_train_y.head().to_markdown())

In [ ]:
print(example_test_x.info())
print(example_test_x.sort_values(by='location').head(8).to_markdown())

In [ ]:
print(ilinet_hhs.info())
print(ilinet_hhs.head().to_markdown())

In [ ]:
print(ilinet.info())
print(ilinet.head().to_markdown())

In [ ]:
print(ilinet_state.info())
print(ilinet_state.head().to_markdown())

# Begin mutable cells

In [ ]:
# Describe the core principles of the required method:
# -
# -
# -

from typing import Any
import numpy as np
import pandas as pd

def fit_and_predict_fn(
    train_x: pd.DataFrame,
    train_y: pd.Series,
    test_x: pd.DataFrame,
) -> pd.DataFrame:
  """Make predictions for test_x using the required method by modelling train_x to train_y. Return quantiles.

  Do not do any cross-validation in here.
  """
  return


# End mutable cells

In [ ]:
### Evaluation helper functions

from typing import Callable, Set, Tuple
import warnings
import numpy as np
import pandas as pd


# --- Scoring Functions ---
def interval_score(
    observed: float, lower: float, upper: float, alpha: float
) -> float:
  if observed < lower:
    score = (upper - lower) + (2 / alpha) * (lower - observed)
  elif observed > upper:
    score = (upper - lower) + (2 / alpha) * (observed - upper)
  else:
    score = upper - lower
  return score


def weighted_interval_score(
    predicted: pd.DataFrame,
    observed: pd.Series,
    quantile_level=QUANTILES,
    count_median_twice=True,
) -> np.array:
  """WIS is a proper scoring rule to evaluate forecasts in an interval or quantile format.

  Smaller values are better. (See Bracher et al., 2021)

  Args:
      observed: Numeric data of n observed values.
      predicted: Numeric nxN dataframe of predictive quantiles with n forecasts
        (same as number of observed values) and N quantiles per forecast. If
        `observed` is a single number, then predicted can just be an N-vector.
      quantile_level (np.array or list): Vector of of size N with the quantile
        levels for which predictions were made.
      count_median_twice (bool, optional): If True, count the median twice in
        the score. Defaults to False.

  Returns:
      np.array: a numeric vector with WIS values of size n (one per observation)
  """

  # If 'observed' is a single value (median) (WIS simplifies to absolute error)
  if observed.size == 1:
    predicted = predicted.reshape(1, -1)

  first_half_quantiles = sorted(quantile_level)[: len(quantile_level) // 2]
  K = len(first_half_quantiles)
  # WIS for each observation
  wis_scores = []
  for i in range(len(observed)):
    y = observed.iloc[i]
    predictions = predicted.iloc[i]
    weighted_interval_scores = []
    median_forecast = predictions.iloc[predictions.shape[0] // 2]
    weighted_interval_scores.append(0.5 * abs(y - median_forecast))
    if count_median_twice:
      weighted_interval_scores.append(0.5 * abs(y - median_forecast))

    for l_q in first_half_quantiles:
      # l: the lower bound of the interval
      # u: the upper bound of the interval
      u_q = round(1 - l_q, 3)
      alpha_i = 2 * l_q
      weighted_interval_scores.append(
          (alpha_i / 2)
          * (
              interval_score(
                  y,
                  predictions[f'quantile_{l_q}'],
                  predictions[f'quantile_{u_q}'],
                  alpha_i,
              )
          )
      )
    # Weighted sum to get WIS score for observation
    wis_scores.append(np.sum(np.array(weighted_interval_scores)))
  return 1 / (K + 0.5) * np.array(wis_scores)


# --- Rolling Window Evaluation Framework ---


def get_saturdays_between_dates(start_date_str: str, end_date_str: str) -> list:
  """Generates a list of Saturday dates within a given date range."""
  date_range = pd.date_range(
      start=start_date_str, end=end_date_str, freq='D', inclusive='both'
  )
  saturdays = date_range[date_range.weekday == 5]
  return [d for d in saturdays] # Return datetime objects


def _process_single_fold(
    reference_date: pd.Timestamp,
    observed_values: pd.DataFrame,
    fit_and_predict_fn: Callable,
    horizons: list,
    location_codes: np.ndarray,
    locations_df: pd.DataFrame,
    encountered_warnings: Set[str],
) -> pd.DataFrame:
  """Processes a single fold: prepares data, runs prediction, and returns the forecast."""
  train_end_date = reference_date - pd.Timedelta(weeks=1)
  # Ensure target_end_date is datetime for comparison
  observed_values['target_end_date'] = pd.to_datetime(observed_values['target_end_date'])
  train = observed_values[observed_values['target_end_date'] <= train_end_date]
  train_x = train.drop(columns=[TARGET_STR])
  train_y = train[TARGET_STR]

  if train_x.empty or train_y.empty:
    print(
        f'Skipping fold for reference_date {reference_date} due to empty'
        ' training data.'
    )
    return pd.DataFrame()  # Return empty DF to signify a skipped fold

  test_x_i = _create_fold_scaffold(
      reference_date, horizons, location_codes, locations_df
  )

  with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    predicted_quantiles = fit_and_predict_fn(
        train_x.copy(), train_y.copy(), test_x_i.copy()
    )
    for warning_message in w:
      msg = str(warning_message.message)
      if msg not in encountered_warnings:
        print(f'Encountered new warning: {msg}')
        encountered_warnings.add(msg)

  forecast_df = test_x_i.copy()
  for col in predicted_quantiles.columns:
    forecast_df[col] = predicted_quantiles[col]

  return forecast_df


def compute_rolling_evaluation(
    observed_values: pd.DataFrame,
    reference_dates: list,
    fit_and_predict_fn: Callable,
    horizons: list,
    location_codes: np.ndarray,
    locations_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, float]:
  """Generates forecasts and computes scores for a set of reference dates.

  Always returns a complete forecast DataFrame. The score will be NaN if no
  ground truth is available for scoring.
  """
  all_forecasts = []
  fold_scores = []
  encountered_warnings = set()

  # Ensure target_end_date is datetime for comparison
  observed_values['target_end_date'] = pd.to_datetime(observed_values['target_end_date'])


  for ref_date in reference_dates:
    forecast_df = _process_single_fold(
        ref_date,
        observed_values,
        fit_and_predict_fn,
        horizons,
        location_codes,
        locations_df,
        encountered_warnings,
    )
    if forecast_df.empty:
      continue

    all_forecasts.append(forecast_df)

    # Try to score by merging with ground truth
    data_for_scoring = pd.merge(
        forecast_df,
        observed_values[['target_end_date', 'location', TARGET_STR]],
        how='inner',  # Inner join filters to only rows with ground truth
        on=['target_end_date', 'location'],
    )

    if not data_for_scoring.empty:
      y_observed = data_for_scoring[TARGET_STR]
      y_predicted = data_for_scoring[[f'quantile_{q}' for q in QUANTILES]]
      scores_in_fold = weighted_interval_score(y_predicted, y_observed)
      fold_scores.append(np.mean(scores_in_fold))

  if not all_forecasts:
    print('No forecasts were generated. Returning an empty DataFrame.')
    return pd.DataFrame(), np.nan

  # Calculate final score, handling the case where no folds could be scored
  final_score = np.mean(fold_scores) if fold_scores else np.nan

  # Combine all forecast dataframes
  final_forecasts_df = pd.concat(all_forecasts, ignore_index=True)

  return final_forecasts_df, final_score

In [ ]:
# Plotting functions

import matplotlib.dates as mdates
import matplotlib.pyplot as plt


def plot_season_forecasts(
    season_start: str,
    season_end: str,
    season_name: str,
    all_forecasts_df: pd.DataFrame,
    national_observed_df: pd.DataFrame,
    step_size: int = 4,
):
  """Generates and displays a consolidated forecast plot for a single flu season.

  Args:
      season_start (str): The start date of the season (e.g., '2023-10-15').
      season_end (str): The end date of the season (e.g., '2024-05-15').
      season_name (str): The name of the season for the plot title (e.g.,
        '2023-2024').
      all_forecasts_df (pd.DataFrame): DataFrame containing all forecast data.
      national_observed_df (pd.DataFrame): Pre-aggregated DataFrame of national
        observed data.
      step_size (int): The interval for plotting forecast plumes (e.g., 4 means
        every 4th forecast).
  """
  # --- 1. Filter data for the specified season ---
  season_forecasts_df = all_forecasts_df[
      (all_forecasts_df['reference_date'] >= season_start)
      & (all_forecasts_df['reference_date'] <= season_end)
  ].copy()

  national_observed_season = national_observed_df[
      (national_observed_df['target_end_date'] >= season_start)
      & (national_observed_df['target_end_date'] <= season_end)
  ]

  # --- 2. Plotting ---
  plt.style.use('seaborn-v0_8-whitegrid')
  fig, ax = plt.subplots(figsize=(16, 9))

  # Plot observed data
  ax.plot(
      national_observed_season['target_end_date'],
      national_observed_season['Total Influenza Admissions'],
      color='black',
      marker='o',
      linestyle='-',
      linewidth=2,
      label='Observed Data',
  )

  # Get unique forecast dates and apply the step size
  unique_reference_dates = sorted(
      season_forecasts_df['reference_date'].unique()
  )
  dates_to_plot = unique_reference_dates[::step_size]

  quantile_cols = [f'quantile_{q}' for q in QUANTILES]

  # Loop through each selected reference date and plot its forecast plume
  for i, forecast_date in enumerate(dates_to_plot):
    single_forecast_df = season_forecasts_df[
        season_forecasts_df['reference_date'] == forecast_date
    ]

    if single_forecast_df.empty:
        print(f"Warning: No forecast data for reference date {forecast_date}. Skipping plot for this date.")
        continue

    # Explicitly select numeric and quantile columns for summation, ADD 'target_end_date'
    cols_to_sum = ['target_end_date', 'horizon', 'location', 'population'] + quantile_cols
    existing_cols_to_sum = [col for col in cols_to_sum if col in single_forecast_df.columns]

    if not existing_cols_to_sum:
        print(f"Warning: No numeric/quantile cols found for reference date {forecast_date}. Skipping plot for this date.")
        continue

    national_forecast = single_forecast_df[existing_cols_to_sum].groupby('target_end_date').sum()

    # Check if national_forecast is empty or missing quantile columns
    if national_forecast.empty or not all(q_col in national_forecast.columns for q_col in quantile_cols):
        print(f"Warning: National forecast for reference date {forecast_date} is empty or missing quantile columns after aggregation. Skipping plot for this date.")
        continue

    # Check quantile columns are numeric
    for q_col in quantile_cols:
        if q_col in national_forecast.columns:
            national_forecast[q_col] = pd.to_numeric(national_forecast[q_col], errors='coerce')
            national_forecast[q_col] = national_forecast[q_col].fillna(0)

    is_first_forecast = i == 0

    # Plot Prediction Intervals
    ax.fill_between(
        mdates.date2num(national_forecast.index),
        national_forecast['quantile_0.025'],
        national_forecast['quantile_0.975'],
        color='steelblue',
        alpha=0.1,
        label='95% Prediction Interval' if is_first_forecast else None,
    )
    ax.fill_between(
        mdates.date2num(national_forecast.index),
        national_forecast['quantile_0.1'],
        national_forecast['quantile_0.9'],
        color='steelblue',
        alpha=0.2,
        label='80% Prediction Interval' if is_first_forecast else None,
    )
    ax.fill_between(
        mdates.date2num(national_forecast.index),
        national_forecast['quantile_0.25'],
        national_forecast['quantile_0.75'],
        color='steelblue',
        alpha=0.4,
        label='50% Prediction Interval' if is_first_forecast else None,
    )

    # Plot Median Forecast line
    ax.plot(
        mdates.date2num(national_forecast.index),
        national_forecast['quantile_0.5'],
        color='tab:blue',
        marker='o',
        linestyle='-',
        linewidth=2.0,
        label='TS Forecast' if is_first_forecast else None,
    )

  # --- 3. Final Formatting and Labels ---
  ax.set_title(
      f'National Flu Hospitalizations Forecasts ({season_name} Season)',
      fontsize=18,
  )
  ax.set_xlabel('Date', fontsize=12)
  ax.set_ylabel('Weekly Hospital Admissions', fontsize=12)
  ax.legend(loc='upper right', frameon=True, edgecolor='black')

  # Format the x-axis to show month abbreviations
  ax.xaxis.set_major_locator(mdates.MonthLocator())
  ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
  plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

  # Set axis limits
  ax.set_xlim(pd.to_datetime(season_start), pd.to_datetime(season_end))
  ax.set_ylim(bottom=0)

  plt.tight_layout()
  plt.savefig(season_name + '.png')
  plt.show()

### Execute Validation Run

In [ ]:
# [exclude_from_prompt]

# --- Execute Validation Run ---
print("--- Starting Validation Run ---")
# Define validation and test periods
flu_season_23_24_val = get_saturdays_between_dates('2023-10-13', '2024-05-15')
flu_season_24_25_val = get_saturdays_between_dates('2024-10-15', '2025-05-15')
validation_reference_dates = flu_season_23_24_val + flu_season_24_25_val
validation_forecasts, validation_score = compute_rolling_evaluation(
    observed_values=dataset.copy(),
    reference_dates=validation_reference_dates,
    fit_and_predict_fn=fit_and_predict_fn,
    horizons=horizons,
    location_codes=location_codes,
    locations_df=locations,
)


print(f'Candidate metric to be minimized: {validation_score}')

if not validation_forecasts.empty:
    validation_forecasts.to_csv('validation_forecasts.csv', index=False)
    print("Validation forecasts saved to 'validation_forecasts.csv'")

In [ ]:
# [exclude_from_prompt]

# Plot forecast on the first validation season

validation_forecasts['target_end_date'] = pd.to_datetime(
    validation_forecasts['target_end_date']
)
validation_forecasts['reference_date'] = pd.to_datetime(
    validation_forecasts['reference_date']
)

# Prepare the observed data
national_observed_all = (
    dataset.groupby('target_end_date')['Total Influenza Admissions']
    .sum()
    .reset_index()
)
national_observed_all['target_end_date'] = pd.to_datetime(
    national_observed_all['target_end_date']
)


seasons_to_plot = [
    {'start': '2023-10-15', 'end': '2024-05-15', 'name': '2023-2024'},
]

for season in seasons_to_plot:
  print(f"--- Generating plot for {season['name']} season ---")
  plot_season_forecasts(
      season_start=season['start'],
      season_end=season['end'],
      season_name=season['name'],
      all_forecasts_df=validation_forecasts,
      national_observed_df=national_observed_all,
      step_size=1,  # Plot every 2nd forecast plume
  )

In [ ]:
# [exclude_from_prompt]

# Plot forecast on the second validation season

seasons_to_plot = [
    {'start': '2024-10-15', 'end': '2025-05-15', 'name': '2024-2025'},
]

for season in seasons_to_plot:
    print(f"--- Generating plot for {season['name']} season ---")
    plot_season_forecasts(
        season_start=season['start'],
        season_end=season['end'],
        season_name=season['name'],
        all_forecasts_df=validation_forecasts,
        national_observed_df=national_observed_all,
        step_size=1
    )

### Execute Test Run

In [ ]:
# [exclude_from_prompt]

# --- Execute Test Run ---
print("\n--- Starting Test Run ---")
# We'll generate forecasts for a different flu season to act as a "test" set.
flu_season_22_23_test = get_saturdays_between_dates("2022-10-15", "2023-05-15")
test_forecasts, test_score = compute_rolling_evaluation(
    observed_values=dataset.copy(),
    reference_dates=flu_season_22_23_test,
    fit_and_predict_fn=fit_and_predict_fn,
    horizons=horizons,
    location_codes=location_codes,
    locations_df=locations,
)

print(
    f"\nTest Score: {test_score}"
)  # This will be NaN if test dates are in the future
if not test_forecasts.empty:
  test_forecasts.to_csv("test_forecasts.csv", index=False)
  print("Test forecasts saved to 'test_forecasts.csv'")

In [ ]:
# [exclude_from_prompt]

# Plot forecast on the test season

test_forecasts['target_end_date'] = pd.to_datetime(
    test_forecasts['target_end_date']
)
test_forecasts['reference_date'] = pd.to_datetime(
    test_forecasts['reference_date']
)

# Prepare the observed data
national_observed_all = (
    dataset.groupby('target_end_date')['Total Influenza Admissions']
    .sum()
    .reset_index()
)
national_observed_all['target_end_date'] = pd.to_datetime(
    national_observed_all['target_end_date']
)

seasons_to_plot = [
    {'start': '2022-10-15', 'end': '2023-05-15', 'name': '2022-2023'},
]

for season in seasons_to_plot:
  print(f"--- Generating plot for {season['name']} season ---")
  plot_season_forecasts(
      season_start=season['start'],
      season_end=season['end'],
      season_name=season['name'],
      all_forecasts_df=test_forecasts,
      national_observed_df=national_observed_all,
      step_size=2,
  )